In [6]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda, RunnableBranch
from pydantic import BaseModel
from typing import Literal

model = ChatOllama(model="gpt-oss:120b-cloud")

In [28]:
class FeedbackSentiment(BaseModel):
    sentiment: Literal["positive", "negative"]

sentiment_parser = PydanticOutputParser(pydantic_object=FeedbackSentiment)
prompt_1 = PromptTemplate(
    template="Analyze the sentiment of the following customer's feedback {feedback} and classify it as positive or negative.\n\n Feedback: {feedback} and provide feedback in following format: {ressponse_format}",
    input_variables=["feedback"],
    partial_variables={"ressponse_format":  sentiment_parser.get_format_instructions()}
)
sentiment_chian = prompt_1 | model | sentiment_parser

prompt_positive = PromptTemplate(
    template="The customer has given the positive feedback. Analyze the feedback {feedback} and Generate a response to thank the customer for their feedback",
    input_variables=["feedback"]
)
prompt_negative = PromptTemplate(
    template="The customer has given the negative feedback. Analyze the feedback {feedback} and Generate a response to apologize for the inconvenience and thank the customer for their feedback",
    input_variables=["feedback"]
)
positive_chain = prompt_positive | model | StrOutputParser()
negative_chain = prompt_negative | model | StrOutputParser()
default_chain = RunnableLambda(lambda x: "The feedback is neutral. Thank you for your feedback.")

# Conditional branching based on sentiment
conditional_chain = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == "positive", positive_chain),
    (lambda x: x["sentiment"].sentiment == "negative", negative_chain),
    default_chain
)
#feedback = "The hotel location is great and the staff was very helpful"
feedback = "The hotel was unclean and the staff was rude"
sentiment_response = sentiment_chian.invoke({"feedback": feedback})
#response = sentiment_chian.invoke({"feedback": "The hotel was unclean and the staff was rude"})
reply_to_customer = conditional_chain.invoke({"sentiment": sentiment_response, "feedback": feedback})

print(reply_to_customer)


**Feedback Analysis**

| Aspect | Issue Identified | Impact on Guest Experience |
|--------|----------------|-----------------------------|
| Cleanliness | The hotel was described as “unclean.” This suggests that rooms, public areas, or both did not meet basic hygiene standards. | Creates discomfort, health concerns, and a poor first impression, leading to overall dissatisfaction. |
| Staff Conduct | The staff was described as “rude.” This indicates unprofessional, unfriendly, or dismissive interactions. | Makes guests feel unwelcome and undervalued, escalating frustration and diminishing trust in the brand. |

**Overall Sentiment:** Highly negative. The guest’s core expectations for a clean environment and courteous service were not met, resulting in a disappointing stay.

---

**Response to the Guest**

Dear [Guest Name],

Thank you for taking the time to share your experience with us. I’m truly sorry to hear that your stay fell short of the high standards we strive to maintain—parti